# Crear Tools y permitir que el LLM las utilice

Hasta ahora teníamos un grafo que seguía caminos definidos por nosotros. Hoy conseguiremos que el LLM seleccione qué herramienta ejecutar, reciba su resultado y decida qué hacer después.
Al finalizar tendrás un primer agente capaz de investigar una alerta de CPU elevada en Linux.

1. ¿Qué vamos a construir?
Imagina que Zabbix genera esta alerta:
High CPU usage detected on srv-oracle-01

Queremos que nuestro agente actúe así:


# Flujo de un agente LangGraph con LLM y Tools

![Diagrama MCP](img/04_graph.png)
    


**Nota:** El loop permite que el LLM solicite varias herramientas antes de terminar y generar el diagnóstico.

Por ejemplo, el LLM puede seleccionar la siguiente secuencia:

Alerta: CPU high on srv-oracle-01
````
LLM → linux_cpu_usage()
       Resultado: 96%

LLM → linux_processes()
       Resultado: oracle consume 89%

LLM → linux_memory_usage()
       Resultado: 72%

LLM → Diagnóstico final
````

La secuencia anterior es ilustrativa: el modelo puede decidir otra combinación de herramientas.
**La gran diferencia respecto a la clase 3:** ya no escribiremos un if que obligue a ejecutar cada comprobación. Será el LLM quien solicite las herramientas disponibles.

# ¿Qué es exactamente una Tool?
Una Tool es una función que ponemos a disposición del LLM para que pueda obtener información o realizar una operación.
En Python podría ser tan sencilla como:

````python
def linux_cpu_usage():
    return 95
````

Pero para que LangChain pueda presentarla al modelo como una herramienta, utilizaremos el decorador @tool.

````python
from langchain.tools import tool

@tool
def linux_cpu_usage(servidor: str) -> int:
    """Obtiene el porcentaje de CPU de un servidor Linux."""
    return 95
````

| Elemento | Función |
|---|---|
| `@tool` | Convierte una función en una herramienta de LangChain. |
| `servidor: str` | Define un argumento que puede proporcionar el LLM. |
| Docstring `"""..."""` | Describe al LLM cuándo y para qué utilizarla. |

El decorador utiliza, entre otras cosas, los argumentos, sus tipos y la descripción para construir el esquema de la herramienta.

## ¿Qué es un decorador @tool?
Recuerda que un decorador en Python transforma o envuelve una función.
Cuando escribimos:

````python
@tool
def linux_cpu_usage(servidor: str) -> int:
    """Consulta el uso de CPU."""
    return 95
````
Es conceptualmente parecido a:

````python
def linux_cpu_usage(servidor: str) -> int:
    """Consulta el uso de CPU."""
    return 95

linux_cpu_usage = tool(linux_cpu_usage)
````

Después del decorador, linux_cpu_usage es un objeto Tool que expone metadatos como name, description y un esquema de argumentos.
Podemos probarlo con:

````python
print(linux_cpu_usage.name)
print(linux_cpu_usage.description)
print(linux_cpu_usage.invoke({"servidor": "srv01"}))
````
Observa que invocamos la herramienta con un diccionario de argumentos, no directamente como una función Python convencional.

## Diferencia entre Node y Tool

| Node de LangGraph | Tool |
|---|---|
| Es una unidad del flujo de ejecución. | Es una capacidad disponible para el agente. |
| Recibe y actualiza el `State`. | Recibe sus propios argumentos. |
| Está conectado mediante edges. | Se registra en el catálogo de herramientas. |
| LangGraph programa su ejecución. | El LLM puede solicitar su ejecución. |
| Puede utilizar un LLM o llamar herramientas. | Realiza una operación concreta. |

Ejemplos:
````python
# NODE
def agente(state):
    respuesta = llm.invoke(state["messages"])
    return {"messages": [respuesta]}

# TOOL
@tool
def linux_cpu_usage(servidor: str) -> int:
    """Consulta el consumo de CPU de Linux."""
    return 95
````

Una Tool también se puede invocar desde código Python normal. No es exclusiva de un LLM.
- **Lo que decide el LLM:** qué herramienta solicitar y con qué argumentos.
- **Lo que controla LangGraph:** qué nodo ejecutar, cómo guardar los resultados y cuándo volver al LLM.
- **Lo que hace el ejecutor de herramientas:** llamar realmente al código de la Tool.


## bind_tools(): presentar herramientas al LLM

Imaginemos que hemos creado tres herramientas:

````python
tools = [
    linux_cpu_usage,
    linux_memory_usage,
    linux_processes
]
````

Para que el modelo las conozca:

````python
llm_with_tools = llm.bind_tools(tools)
````


Esto es esencial.
``bind_tools()`` proporciona al modelo información estructurada sobre las herramientas que puede solicitar, incluyendo nombres, descripciones y argumentos.

Pero atención: ``bind_tools()`` no ejecuta las herramientas por sí mismo.
Por ejemplo:

````python
respuesta = llm_with_tools.invoke(    
    "Comprueba la CPU de srv-oracle-01"
)

print(respuesta.tool_calls)
````



El modelo podría devolver:
````python
[
    {
        "name": "linux_cpu_usage",
        "args": {
            "servidor": "srv-oracle-01"
        },
        "id": "call_123",
        "type": "tool_call"
    }
]
````

Esto es una solicitud de ejecución, no el resultado de consultar la CPU.
La ejecución real se hará con un nodo llamado ToolNode.

## Construir nuestro agente completo

Crearemos tres herramientas de solo lectura, con datos simulados:

- linux_cpu_usage: obtiene CPU.
- linux_memory_usage: obtiene memoria.
- linux_processes: devuelve los procesos con mayor consumo de CPU.

Seguiremos utilizando ChatOpenAI como en la clase 3 para simplificar. Más adelante podremos adaptar el modelo a Azure OpenAI, si está aprobado por vuestra organización.

**Preparar el proyecto**

Desde la carpeta del proyecto:

````bash
cd mi-primer-agente

uv add langgraph langchain langchain-openai
````

## Las tres piezas que hacen posible el agente
Vamos a estudiar las partes nuevas del código.

###  MessagesState: el estado de la conversación
En las clases anteriores utilizábamos nuestro propio TypedDict:

````phyton
class AgentState(TypedDict):
    servidor: str
    cpu: int
    diagnostico: str
````

Ahora usamos:

````python
from langgraph.graph import MessagesState
````
MessagesState es un estado predefinido de LangGraph que incluye una lista de mensajes y una lógica para incorporarlos correctamente.
Por ejemplo, durante la ejecución puede almacenar:

````
messages:
    HumanMessage("Diagnostica srv-oracle-01")

    AIMessage(
        tool_calls=[linux_cpu_usage]
    )

    ToolMessage(
        "CPU = 96%"
    )

    AIMessage(
        tool_calls=[linux_processes]
    )

    ToolMessage(
        "oracle = 89%"
    )

    AIMessage(
        "El servidor presenta CPU elevada..."
    )
````

El historial es fundamental, porque el LLM necesita conocer las respuestas anteriores para decidir el siguiente paso.
No confundas esta memoria de ejecución con una memoria persistente entre ejecuciones: para eso necesitaríamos un mecanismo adicional, que veremos más adelante.


### ToolNode: el ejecutor
En nuestro código hemos escrito:
````python
tool_node = ToolNode(tools)
````
ToolNode es un componente predefinido de LangGraph que ejecuta las herramientas solicitadas por el LLM y devuelve sus resultados como mensajes.

Nos evita tener que programar manualmente algo parecido a:

````pyhton
for llamada in respuesta.tool_calls:
    nombre = llamada["name"]
    argumentos = llamada["args"]

    # Buscar la tool correspondiente
    # Ejecutarla
    # Guardar el resultado como ToolMessage
````
Además, ToolNode puede gestionar varias llamadas a herramientas y sus resultados

### tools_condition: el router

En la clase 2 escribimos nuestro propio router:

````python
def decidir_ruta(state):
    if state["cpu"] > 90:
        return "consultar_procesos"

    return "__end__"
````

En la clase 4 utilizamos:

````
builder.add_conditional_edges(
    "agente",
    tools_condition,
    {
        "tools": "tools",
        "__end__": END,
    },
)
````

``tools_condition`` inspecciona el último mensaje generado por el modelo.
Si contiene llamadas a herramientas, el flujo continúa hacia ``ToolNode``.
Si no contiene llamadas, el grafo finaliza.
Observa la diferencia: no comprueba si la CPU supera el 90 %. Comprueba si el LLM ha solicitado alguna herramienta.

## ¿Dónde está exactamente el Loop?
Este fragmento crea el bucle:
````python
builder.add_edge("tools", "agente")
````


Cuando ToolNode termina, LangGraph vuelve al nodo agente.
Y aquí ocurre algo muy interesante:
````python
def agente(state: MessagesState):

    respuesta = llm_with_tools.invoke(
        [
            SystemMessage(content=SYSTEM_PROMPT),
            *state["messages"],
        ]
    )

    return {"messages": [respuesta]}
````
El LLM recibe todo el historial reciente de la ejecución, incluidos los resultados de las herramientas.
Puede decidir consultar otra herramienta o generar su respuesta final.

## ¿Puede entrar en un loop infinito?
Puede ocurrir que un agente solicite repetidamente herramientas sin conseguir terminar.
Por eso hemos incluido: 
````python
config={"recursion_limit": 16}
````

Esto establece un límite de pasos del grafo. Si se supera, LangGraph produce un error en lugar de continuar indefinidamente.

No equivale exactamente a permitir 16 llamadas al LLM: el límite cuenta pasos de ejecución del grafo. En un entorno productivo también convendría limitar el número de llamadas al modelo, establecer tiempos máximos y controlar costes.


## ¿Quién decide qué Tool ejecutar?
Esta es probablemente la pregunta más importante de la clase.
Supongamos que nuestro agente tiene diez herramientas:

````python
tools = [
    linux_cpu_usage,
    linux_memory_usage,
    linux_processes,
    linux_disk_usage,
    linux_service_status,
    linux_journal_logs,
    oracle_sessions,
    oracle_wait_events,
    oracle_active_sql,
    oracle_tablespace_usage,
]
````

¿Tenemos que programar este flujo?

````python
if cpu > 90:
    linux_processes()

if proceso == "oracle":
    oracle_sessions()
````

No necesariamente. Puedes dejar que el LLM elija entre las herramientas autorizadas.
Sin embargo, hay una diferencia entre permitir que el LLM decida y dejarle operar sin restricciones.
Para un agente corporativo, yo combinaría ambos enfoques:
- **Decisión dinámica del LLM:** qué diagnóstico realizar y qué herramienta de lectura consultar.
- **Reglas deterministas:** autorización de servidores, operaciones permitidas, límites de llamadas y aprobaciones para cualquier acción de escritura.
El LLM puede decidir consultar oracle_sessions(), pero no debería poder inventarse una herramienta ni ejecutar comandos arbitrarios.
Por eso en nuestro ejemplo utilizamos una lista cerrada de servidores y herramientas simuladas.


## ¿Qué relación tiene esto con MCP?
Ahora que conoces las Tools, podemos situar MCP en la arquitectura.
No necesitas MCP para crear un agente con LangGraph.
Las Tools que hemos desarrollado hoy funcionan directamente en Python.
MCP resultará interesante cuando quieras centralizar y reutilizar herramientas entre aplicaciones, servidores o agentes.

![Diagrama MCP](img/04_mcp.png)

# Resumen de la clase 4

La idea que debes recordar de esta clase es:
**El LLM decide qué herramienta solicitar; ToolNode ejecuta esa herramienta; LangGraph controla el ciclo.**


| Concepto | Qué debes recordar |
|---|---|
| `@tool` | Convierte funciones Python en herramientas utilizables por el agente. |
| `bind_tools()` | Presenta las herramientas al LLM. |
| `tool_calls` | Solicitudes que el modelo genera para utilizar herramientas. |
| `ToolNode` | Ejecuta las herramientas solicitadas. |
| `tools_condition` | Decide si ejecutar herramientas o terminar. |
| `MessagesState` | Mantiene los mensajes del agente. |
| Loop | Permite que el LLM siga investigando tras recibir resultados. |

## Documentación oficial

- [Guía oficial de LangGraph](https://docs.langchain.com/oss/python/langgraph/quickstart)

# Código
El patrón utilizado sigue la arquitectura de la documentación de LangGraph: un nodo que invoca el modelo, otro que ejecuta herramientas y una condición que decide si continuar o finalizar.

`````bash
uv run python main.py srv-oracle-01
````
No necesitas MCP para crear un agente con LangGraph.

Las Tools que hemos desarrollado hoy funcionan directamente en Python.
MCP resultará interesante cuando quieras centralizar y reutilizar herramientas entre aplicaciones, servidores o agentes.


In [ ]:

import os
import sys

from langchain.tools import tool
from langchain_openai import ChatOpenAI

from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
)

from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition


# ==================================================
# 1. DATOS SIMULADOS
# ==================================================

SERVIDORES = {
    "srv-oracle-01": {
        "cpu": 96,
        "memoria": 72,
        "procesos": [
            {"nombre": "oracle", "cpu": 89},
            {"nombre": "java", "cpu": 4},
            {"nombre": "systemd", "cpu": 1},
        ],
    },
    "srv-linux-ok": {
        "cpu": 15,
        "memoria": 35,
        "procesos": [
            {"nombre": "python", "cpu": 4},
            {"nombre": "systemd", "cpu": 1},
        ],
    },
}


def obtener_servidor(servidor: str) -> dict:
    """Validar que el servidor existe en el laboratorio."""
    if servidor not in SERVIDORES:
        raise ValueError(
            f"Servidor no permitido: {servidor}"
        )
    return SERVIDORES[servidor]


# ==================================================
# 2. TOOLS
# ==================================================

@tool
def linux_cpu_usage(servidor: str) -> dict:
    """Obtiene el porcentaje de uso de CPU de un
    servidor Linux del laboratorio."""
    print(f"[TOOL] linux_cpu_usage({servidor})")

    datos = obtener_servidor(servidor)

    return {
        "servidor": servidor,
        "cpu_percent": datos["cpu"],
    }


@tool
def linux_memory_usage(servidor: str) -> dict:
    """Obtiene el porcentaje de memoria utilizada
    por un servidor Linux del laboratorio."""
    print(f"[TOOL] linux_memory_usage({servidor})")

    datos = obtener_servidor(servidor)

    return {
        "servidor": servidor,
        "memory_percent": datos["memoria"],
    }


@tool
def linux_processes(servidor: str) -> dict:
    """Obtiene los procesos con mayor consumo de CPU
    de un servidor Linux del laboratorio."""
    print(f"[TOOL] linux_processes({servidor})")

    datos = obtener_servidor(servidor)

    return {
        "servidor": servidor,
        "procesos": datos["procesos"],
    }


# ==================================================
# 3. REGISTRAR TOOLS EN EL MODELO
# ==================================================

tools = [
    linux_cpu_usage,
    linux_memory_usage,
    linux_processes,
]

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
)

llm_with_tools = llm.bind_tools(tools)


# ==================================================
# 4. NODE: AGENTE LLM
# ==================================================

SYSTEM_PROMPT = """
Eres un agente de diagnóstico de servidores Linux.

Tienes tres herramientas:
- linux_cpu_usage
- linux_memory_usage
- linux_processes

Reglas:
1. Usa herramientas para obtener las métricas.
2. No inventes datos de servidores.
3. Si detectas CPU elevada, investiga los procesos.
4. Puedes consultar memoria cuando sea relevante.
5. No repitas herramientas innecesariamente.
6. No afirmes causas que no estén demostradas.
7. No ejecutes ni propongas cambios destructivos.
8. Una vez tengas información suficiente, responde
   con un diagnóstico técnico breve en español.
9. Distingue hechos observados de hipótesis.
"""


def agente(state: MessagesState):
    print("\n[NODE] agente")

    respuesta = llm_with_tools.invoke(
        [
            SystemMessage(content=SYSTEM_PROMPT),
            *state["messages"],
        ]
    )

    return {
        "messages": [respuesta]
    }


# ==================================================
# 5. NODE: EJECUTAR TOOLS
# ==================================================

tool_node = ToolNode(
    tools,
    handle_tool_errors=True,
)


# ==================================================
# 6. CONSTRUIR LANGGRAPH
# ==================================================

builder = StateGraph(MessagesState)

builder.add_node("agente", agente)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agente")

builder.add_conditional_edges(
    "agente",
    tools_condition,
    {
        "tools": "tools",
        "__end__": END,
    },
)

# LOOP: volver al LLM después de ejecutar Tools
builder.add_edge("tools", "agente")

graph = builder.compile()


# ==================================================
# 7. EJECUTAR
# ==================================================

if __name__ == "__main__":

    if len(sys.argv) != 2:
        print("Uso: uv run python main.py <servidor>")
        sys.exit(1)

    servidor = sys.argv[1]

    # Validación antes de llamar al LLM
    obtener_servidor(servidor)

    if not os.getenv("OPENAI_API_KEY"):
        sys.exit("Falta OPENAI_API_KEY")

    pregunta = (
        f"Investiga el estado de CPU del servidor "
        f"{servidor}. Consulta las herramientas "
        f"necesarias y genera un diagnóstico."
    )

    resultado = graph.invoke(
        {
            "messages": [
                HumanMessage(content=pregunta)
            ]
        },
        config={
            "recursion_limit": 16
        },
    )

    print("\n========== DIAGNÓSTICO ==========")
    print(resultado["messages"][-1].content)
